# Spatial colocalization — concept & motivation

Before we dive into code, let's take a step back and think about what we're actually doing and why.

---

## What is a brain map?

Modern neuroimaging lets us measure all sorts of things at the level of individual brain regions: fMRI activation, cortical thickness, grey matter volume, receptor densities from PET, gene expression from post-mortem tissue, and much more. All of these can be represented as a **brain map** — essentially a vector of numbers, one per brain region, voxel, or vertex.

When we use a parcellation (a predefined set of brain regions), each map becomes a row in a dataframe, and each column is one region. That's the fundamental data structure in NiSpace.

---

## What is spatial colocalization?

The core idea is simple: **if two brain maps overlap spatially, the corresponding biological processes might be related.**

For a concrete example, imagine you're running one of those classic pain studies. Your participants received brief but unpleasant thermal stimuli while lying in an fMRI scanner, and you ended up with a T-map showing which brain regions activate during pain perception. You might now wonder: is the spatial distribution of this pain response related to the distribution of opioid receptors in the brain?

You can't measure opioid receptors with fMRI, and PET scans are expensive, but there are publicly available group-average receptor density maps from healthy subjects at rest. The question then becomes: **does the spatial pattern of pain activation correlate with the spatial pattern of opioid receptor density across brain regions?**

If yes — if regions with high opioid receptor density also show strong pain-related activation — that's what we call a spatial colocalization. It's a correlation, not a proof of causation, but it's a meaningful and interpretable first step.

---

## The spatial autocorrelation problem

Here's the catch: **brain maps are spatially autocorrelated.** Nearby brain regions tend to have similar values, simply because they share anatomical connectivity, developmental history, and functional organization. Two maps that are completely unrelated biologically might still appear correlated just because both happen to show a gradient from occipital to frontal cortex, or both peak in large frontal regions.

In other words, if you just compute a Pearson or Spearman correlation between two brain maps and look up the p-value in a standard table, you'll get inflated significance — the effective sample size is much smaller than the number of brain parcels because the parcels are not independent.

This is not a new problem. It's well-known in spatial statistics as **Tobler's first law of geography**: *everything is related to everything else, but near things are more related than distant things.* Neuroimaging inherited this problem.

---

## How do null models help?

The solution is to build a proper null distribution that accounts for spatial autocorrelation. Rather than relying on parametric assumptions, we generate many **null versions** of the data that preserve the spatial structure of the original maps but break any specific biological relationship — and then ask whether the observed colocalization is more extreme than what the nulls produce.

NiSpace implements three distinct null model paradigms, each suited to a different analysis context.

### 1. Spatial null maps — `permute("maps")`

This is the most direct approach. For each reference map, we generate a set of spatially constrained random versions — maps that have the same overall spatial autocorrelation as the original but scrambled spatial patterns. We then correlate the real input map with thousands of these null maps and compare the observed correlation to that distribution.

Three methods are available:

- **Spin test** (Alexander-Bloch et al., 2018): Projects the parcellation onto a sphere and applies random rotations. The map is literally rotated on the cortical surface, so spatial topology is preserved exactly. Fast and interpretable, but limited to cortical surface parcellations.
- **Moran spectral randomization**: Decomposes the spatial covariance structure into eigenmodes and randomizes the spectral coefficients while preserving the overall structure. Particularly good at capturing global spatial organization; very flexible and quick.
- **Variogram matching** (Burt et al., 2020): Fits a model to how spatial variance grows with distance in the original map, then generates random maps with the same variogram. More flexible — works with any parcellation geometry, not just surface-based ones.

This is the right approach when you have a **single group-level map** and want to test its colocalization with reference maps.

### 2. Group label permutation — `permute("groups")`

When you have **individual subject data** and a group comparison design (patients vs. controls), the null hypothesis is different: *would any arbitrary group division in this dataset produce the same spatial pattern?* Spatial null maps don't answer that — they test whether the specific colocalization could arise from random spatial matching, but they don't account for the fact that the effect size map itself was derived from real subjects with real between-group structure.

The solution is to permute the group labels. For each permutation we randomly shuffle the patient/control assignments, recompute the effect size map from the permuted groups, and then colocalize. The null distribution is thus built from **permuted effect size maps** — not from an explicit spatial model, but from the same data and the same pipeline. This gives the most direct and realistic null for a group comparison design. Group permutation in spatial colocalization analysis was introduced with the JuSpace toolbox ([Dukart et al., 2021](https://doi.org/10.1002/hbm.25244)).

### 3. Set resampling — `permute("sets")` in XSEA

X-Set Enrichment Analysis (XSEA, [Notebook 10](intro10_xsea.ipynb)) tests a different question: not whether a single map colocalizes, but whether the *average colocalization across a set of maps* (e.g., all serotonin receptors combined) is higher than expected. Here there is an additional null hypothesis choice.

One option — and NiSpace's **default** for XSEA — is to still use spatial null maps: apply the same spin/variogram/Moran approach to the input map, recompute the set-level average correlation for each permutation, and compare. This is the most conservative approach and avoids inflation from within-set co-expression. Similarly, the group label permutation can also be applied here.

The alternative is **set resampling**: randomly draw sets of equal size from the full background of reference maps and compute the average colocalization of each random set. There are no null maps at all — the null distribution is built entirely from **resampled colocalization statistics**. This asks whether the observed set mean is higher than random groupings would give. It can be more powerful, but [Fulcher et al. (2021)](https://doi.org/10.1038/s41467-021-22862-1) showed it inflates false positives when maps within a set are highly correlated, which is common in gene expression data. This can be circumvented by adjusting the null sets for within-set co-expression (read, for example, [Wei et al., 2021](https://doi.org/10.1002/hbm.25711)), which is, however, yet to be implemented in NiSpace.

---

All three paradigms are available in NiSpace. We'll go into the practical details in [Notebook 5](intro05_null_models.ipynb) (spatial and group null models) and [Notebook 10](intro10_xsea.ipynb) (XSEA and set resampling).

---

## What NiSpace does

NiSpace provides a complete, flexible toolkit for spatial colocalization analyses. At its core, it does three things:

1. **Parcellation**: Takes your brain images (NIfTI or GIFTI) and parcellates them into a set of regions. Pre-parcellated data (DataFrames) are also accepted.
2. **Colocalization**: Computes pairwise correlations between your maps and a set of reference maps (e.g., neurotransmitter receptor maps, gene expression maps). Several methods are available: Spearman, Pearson, partial correlations, PLS, and more. 
3. **Permutation testing**: Generates null distributions via spatial map permutation, group label permutation, or set resampling, then computes empirical p-values. Multiple comparison correction is included.

On top of that, NiSpace integrates a curated collection of reference datasets and parcellations (PET receptor maps, gene expression, ENIGMA effect sizes, and more), handles data versioning and integrity checking automatically, and provides ready-made visualization tools.

---

## About this series

The following notebooks guide you through NiSpace step by step:

| # | Notebook | What you'll learn |
|---|----------|-------------------|
| 2 | [Getting started: the NiSpace object](intro02_nispace_intro.ipynb) | Core API: `fit()`, `colocalize()`, `permute()`, `correct_p()`, `plot()` |
| 3 | [Data: datasets, parcellations & custom inputs](intro03_fetching_datasets.ipynb) | Fetching reference data, custom parcellations, Parcellater concept |
| 4 | [Working with imaging phenotypes](intro04_imaging_phenotypes.ipynb) | `transform_y()`, `clean_y()`, design matrix |
| 5 | [Null models and permutation testing](intro05_null_models.ipynb) | Spin tests, Moran, Burt, group permutation |
| 6 | [Multiple comparisons correction](intro06_multiple_comparisons.ipynb) | FDR, Bonferroni, Meff, maxT |
| 7 | [Visualizing colocalization results](intro07_visualization.ipynb) | Customizing `NiSpace.plot()` and `NiSpace.plot_brain()` |
| 8 | [Brain plotting with NiSpace](intro08_brainplot.ipynb) | Standalone `brainplot()` and `view_surf()` |
| 9 | [NiSpace workflows](intro09_workflows.ipynb) | One-call wrapper functions for common pipelines |
| 10 | [X-Set Enrichment Analysis](intro10_xsea.ipynb) | XSEA concept, set structure, set-level null models |
| 11 | [Advanced topics](intro11_advanced.ipynb) | Dimensionality reduction, multi-modal parcellations, individual-level colocalization |

Throughout the series we use three example datasets:

- **Pain map** (NeuroQuery meta-analytic z-map for "pain"): a single brain map, perfect for building up the basic workflow.
- **Anorexia nervosa example** (`fetch_example("anorexianervosa")`): **simulated** parcellated grey matter values for 50 patients and 50 healthy controls, used wherever we need individual-subject data and group comparisons. Not intended for scientific use.
- **ENIGMA cortical thickness/area** (`fetch_reference("enigmathick")` / `fetch_reference("enigmaarea")`): effect size maps (Cohen's d) for many psychiatric disorders vs. controls — note that these are *reference* datasets, not example data, but we use them as input maps in the advanced notebooks.

Let's get started.